### Data extraction, cleaning and exploration
The purpose of the following code is to identify valid RollCall numbers (RollCallNumber) that can be used for tracking legislative voting behavior.

The procedure is structured as follows:

+ Initial Data Acquisition – Retrieve all legislative documents from the 194th General Court using the official API endpoint.

+ Data Structuring and Cleaning – Normalize the dataset and conduct exploratory analysis to understand document attributes and identify entries with valid billNumber values, which indicate formally numbered legislative items.

+ Preliminary Filtering – From the cleaned dataset, generate a subset of bill numbers that are theoretically eligible to have RollCall events recorded.

+ Targeted API Requests – Iterate over this filtered list of bill numbers and query the endpoint /Documents/{billNumber}/RollCalls to determine whether a RollCall has occurred and to extract any associated RollCallNumber values.

This selective querying strategy helps reduce unnecessary API calls, ensuring that only legislation with potential recorded votes is processed further.

In [1]:
import requests 

In [2]:
documents194 = requests.get("https://malegislature.gov/api/GeneralCourts/194/Documents/")

In [3]:
documents194.raise_for_status()

In [4]:
docs_json = documents194.json()
print(f"Total documents in General Court 194: {len(docs_json)}")

Total documents in General Court 194: 8716


##### Example of API return value
    [  
      {  
        "billNumber": "string",  
        "docketNumber": "string",  
        "title": "string",
        "primarySponsor": {
        "id": "string",
        "name": "string",
        "type": 1,
        "details": "string",
        "responseDate": "2025-10-12T22:20:55.859Z"
      },
      "cosponsors": [
      {
        "id": "string",
        "name": "string",
        "type": 1,
        "details": "string",
        "responseDate": "2025-10-12T22:20:55.859Z"
      }
    ],
      "jointSponsor": {
        "id": "string",
        "name": "string",
        "type": 1,
        "details": "string",
        "responseDate": "2025-10-12T22:20:55.859Z"
      },
    "generalCourtNumber": 0,
    "details": "string",
    "isDocketBookOnly": "true"
      }
    ]

In [5]:
for i, doc in enumerate(docs_json[:3]):
    print(f"\n--- Document {i+1} preview ---")
    print(doc)


--- Document 1 preview ---
{'BillNumber': 'H3444', 'DocketNumber': 'HD2050', 'Title': 'An Act relative to s-license compliance', 'PrimarySponsor': {'Id': 'JCD1', 'Name': 'James C. Arena-DeRosa', 'Type': 1, 'Details': 'http://malegislature.gov/api/GeneralCourts/194/LegislativeMembers/JCD1', 'ResponseDate': '2025-01-01T10:42:53.773'}, 'Cosponsors': [{'Id': 'JCD1', 'Name': 'James C. Arena-DeRosa', 'Type': 1, 'Details': 'http://malegislature.gov/api/GeneralCourts/194/LegislativeMembers/JCD1', 'ResponseDate': '2025-01-01T10:42:53.773'}], 'JointSponsor': None, 'GeneralCourtNumber': 194, 'Details': 'http://malegislature.gov/api/GeneralCourts/194/Documents/H3444', 'IsDocketBookOnly': False}

--- Document 2 preview ---
{'BillNumber': 'H1283', 'DocketNumber': 'HD3832', 'Title': 'An Act relative to real estate disclosure forms', 'PrimarySponsor': {'Id': 'AHP1', 'Name': 'Alice Hanlon Peisch', 'Type': 1, 'Details': 'http://malegislature.gov/api/GeneralCourts/194/LegislativeMembers/AHP1', 'Response

#### Extract valid bill numbers

In [6]:
#create a list to store bill numbers, docketnumber, title, book only status and branch 
bill_list = []

for doc in docs_json:
    bill_no = doc.get("BillNumber")  
    docket_no = doc.get("DocketNumber")
    title = doc.get("Title")
    is_book_only = doc.get("IsDocketBookOnly")
    #details_url = doc.get("details") 

    bill_list.append({
        "billNumber": bill_no if bill_no else None,
        "docketNumber": docket_no if docket_no else None,
        #"details_url": details_url,
        "title": title if title else None,
        "branch": "Senate" if bill_no and bill_no.startswith("S") else (
            "House" if bill_no and bill_no.startswith("H") else None
        ),
        "isDocketBookOnly": is_book_only
        
    })
    
print(f"Total entries collected (including None fields): {len(bill_list)}")
print("Preview first 10 entries:")
for entry in bill_list[:10]:
    print(entry)


Total entries collected (including None fields): 8716
Preview first 10 entries:
{'billNumber': 'H3444', 'docketNumber': 'HD2050', 'title': 'An Act relative to s-license compliance', 'branch': 'House', 'isDocketBookOnly': False}
{'billNumber': 'H1283', 'docketNumber': 'HD3832', 'title': 'An Act relative to real estate disclosure forms', 'branch': 'House', 'isDocketBookOnly': False}
{'billNumber': 'H1399', 'docketNumber': 'HD3446', 'title': 'An Act relative to an individual Medicare marketplace option', 'branch': 'House', 'isDocketBookOnly': False}
{'billNumber': 'H1282', 'docketNumber': 'HD3438', 'title': 'An Act relative to the Massachusetts Uniform Commercial Code', 'branch': 'House', 'isDocketBookOnly': False}
{'billNumber': 'H1932', 'docketNumber': 'HD3437', 'title': 'An Act relative to the uniform voidable transactions act', 'branch': 'House', 'isDocketBookOnly': False}
{'billNumber': 'H3757', 'docketNumber': 'HD1394', 'title': 'An Act to promote transportation demand management ',

In [7]:
import pandas as pd

df_bills = pd.DataFrame(bill_list)

print("DataFrame created. Shape:", df_bills.shape)
print(df_bills.head())

# count by branch
print("\n Count by branch:")
print(df_bills['branch'].value_counts(dropna=False))


DataFrame created. Shape: (8716, 5)
  billNumber docketNumber                                              title  \
0      H3444       HD2050            An Act relative to s-license compliance   
1      H1283       HD3832    An Act relative to real estate disclosure forms   
2      H1399       HD3446  An Act relative to an individual Medicare mark...   
3      H1282       HD3438  An Act relative to the Massachusetts Uniform C...   
4      H1932       HD3437  An Act relative to the uniform voidable transa...   

  branch  isDocketBookOnly  
0  House             False  
1  House             False  
2  House             False  
3  House             False  
4  House             False  

 Count by branch:
branch
House     4606
Senate    2639
None      1471
Name: count, dtype: int64


In [8]:
df_bills.info

<bound method DataFrame.info of      billNumber docketNumber  \
0         H3444       HD2050   
1         H1283       HD3832   
2         H1399       HD3446   
3         H1282       HD3438   
4         H1932       HD3437   
...         ...          ...   
8711       None       SD3283   
8712      H4606       HD5230   
8713      H4601         None   
8714      S2648       SD3284   
8715       None       HD5231   

                                                  title  branch  \
0               An Act relative to s-license compliance   House   
1       An Act relative to real estate disclosure forms   House   
2     An Act relative to an individual Medicare mark...   House   
3     An Act relative to the Massachusetts Uniform C...   House   
4     An Act relative to the uniform voidable transa...   House   
...                                                 ...     ...   
8711  Department of Revenue August 2025 Net State Ta...    None   
8712  Order relative to extending until Thursda

In [9]:
display(df_bills.describe(include='all'))

,billNumber,docketNumber,title,branch,isDocketBookOnly
count,7245,8455,8716,7245,8716
unique,7245,8455,7306,2,2
top,S2648,HD5231,Emergency Housing Assistance Program bi-weekly...,House,False
freq,1,1,14,4606,8544


In [10]:
df_bills.isna().sum()

billNumber          1471
docketNumber         261
title                  0
branch              1471
isDocketBookOnly       0
dtype: int64

In [11]:
weird_cases = df_bills[
    (df_bills['billNumber'].notna()) & (df_bills['isDocketBookOnly'] == True)
]

print(f"BillNumber exists but still marked as isDocketBookOnly=True: {len(weird_cases)} rows")
display(weird_cases.head())

BillNumber exists but still marked as isDocketBookOnly=True: 0 rows


,billNumber,docketNumber,title,branch,isDocketBookOnly


- A total of 8704 legislative records were retrieved for General Court 194.

- Among them, 7232 entries have formal bill numbers (Hxxxx/Sxxxx), indicating they have been officially numbered and appear in the legislative calendar.

- There are 8532 records marked as isDocketBookOnly=False, meaning they have technically entered the legislative process.⚠️ This implies that approximately 1300 items have no billNumber yet but are still marked as in process (docketNumber-only bills) — these may be early-stage filings not yet formally numbered.

- All of bills with numbers have entered the legislative process.

Therefore, the set of bills suitable for Roll Call tracking should primarily focus on entries where billNumber != None.


#### Extract roll call numbers 

In [12]:
#%pip install tqdm

In [15]:
import requests
import time

#select entries that billNumber is not None
bill_numbers = df_bills[df_bills['billNumber'].notna()]['billNumber'].tolist()[:5000]

rollcall_records = []  # Store {billNumber, rollCallNumbers}
total = len(bill_numbers)

print(f"Starting RollCall scan for first {total} bills...")

for index, bill in enumerate(bill_numbers):
    url = f"https://malegislature.gov/api/Documents/{bill}/RollCalls"
    try:
        resp = requests.get(url, timeout=8)
        resp.raise_for_status()
        data = resp.json()

        if isinstance(data, list) and len(data) > 0:
            roll_nums = [item.get("RollCallNumber") for item in data if "RollCallNumber" in item]
        else:
            roll_nums = []

        rollcall_records.append({
            "billNumber": bill,
            "rollCallNumbers": roll_nums,
            "hasRollCall": len(roll_nums) > 0
        })
    
    except Exception as e:
        rollcall_records.append({
            "billNumber": bill,
            "rollCallNumbers": None,
            "hasRollCall": None
        })
        print(f"⚠ Error fetching {bill}: {e}")

    if (index + 1) % 100 == 0:
        print(f"📌 Progress: {index + 1}/{total} bills processed...")

# convert to DataFrame
df_roll_status_test = pd.DataFrame(rollcall_records)

print("\n RollCall Scan Summary (First 500 Bills):")
print(df_roll_status_test['hasRollCall'].value_counts(dropna=False))

df_roll_status_test.head()



Starting RollCall scan for first 5000 bills...
⚠ Error fetching H3444: 400 Client Error: No Roll Calls could be found for document "H3444" in the current General Court. for url: https://malegislature.gov/api/Documents/H3444/RollCalls
⚠ Error fetching H1283: 400 Client Error: No Roll Calls could be found for document "H1283" in the current General Court. for url: https://malegislature.gov/api/Documents/H1283/RollCalls
⚠ Error fetching H1399: 400 Client Error: No Roll Calls could be found for document "H1399" in the current General Court. for url: https://malegislature.gov/api/Documents/H1399/RollCalls
⚠ Error fetching H1282: 400 Client Error: No Roll Calls could be found for document "H1282" in the current General Court. for url: https://malegislature.gov/api/Documents/H1282/RollCalls
⚠ Error fetching H1932: 400 Client Error: No Roll Calls could be found for document "H1932" in the current General Court. for url: https://malegislature.gov/api/Documents/H1932/RollCalls
⚠ Error fetching H

,billNumber,rollCallNumbers,hasRollCall
0,H3444,None,None
1,H1283,None,None
2,H1399,None,None
3,H1282,None,None
4,H1932,None,None


In [16]:

print(df_roll_status_test['hasRollCall'].value_counts(dropna=False))

hasRollCall
None    5000
Name: count, dtype: int64


In [19]:

print(df_roll_status_test['rollCallNumbers'].value_counts(dropna=False))

rollCallNumbers
None    5000
Name: count, dtype: int64


In [17]:

df_roll_status_test.head()

,billNumber,rollCallNumbers,hasRollCall
0,H3444,None,None
1,H1283,None,None
2,H1399,None,None
3,H1282,None,None
4,H1932,None,None


Based on the result, the actual proportion of bills in the 194th General Court that reached a RollCall stage appears to be very low. We will therefore continue with a more focused extraction approach to identify and retrieve only those bills that have generated RollCall activity.

I am not sure this works, it might just not be able to recognize if it found a roll call or not, there should surely be a rollcall in 5000 bills.